# RT-DETRv2 SportsMOT — test-split performance benchmark (Kaggle T4)

Pulls [`smallTech/rtdetr-sportsmot`](https://huggingface.co/smallTech/rtdetr-sportsmot)
from the Hub and benchmarks its **runtime performance at maximum T4
utilization** on the **full** SportsMOT `test/` split, staged as two mutually
exclusive halves (`testing/prepare-data-1` + `-2` — the whole split exceeds
Kaggle's 20 GB kernel-output cap) and processed one part at a time.

The pipeline saturates the GPU: batched forwards under fp16 autocast with
thread-pool image prefetch. Reported numbers: **throughput** (fps, batched)
and **single-image latency percentiles** (separate batch-1 probe), plus
detections/frame and confidence stats. The test split has **no public ground
truth**, so accuracy metrics (mAP, eval_loss) are impossible here.

Results are logged, written to `benchmarks.json` in the kernel output, and
**published to the Hub model card** (marker-delimited section, updated in
place on re-runs). Run the sibling `smoketest` first.

In [ ]:
# --- 1. Dependencies --------------------------------------------------------
# CRITICAL: do NOT reinstall or upgrade torch OR transformers (torch: GPU
# kernel compatibility; transformers: the checkpoint must load under the
# lineage that trained it). Nothing extra is needed here — performance
# benchmarking uses only the preinstalled stack.
import subprocess, sys, tempfile
import torch, transformers

_con = tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False)
_con.write(f"torch=={torch.__version__}\ntransformers=={transformers.__version__}\n")
_con.close()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-c", _con.name, "codecarbon"],
    check=True,
)
print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
# --- 2. Hugging Face token --------------------------------------------------
# Needed WITH WRITE ACCESS to publish the benchmark section to the model
# card; without it the run still completes and benchmarks.json stays in the
# kernel output. Rides in the private external-secrets dataset (Kaggle
# Secrets are dropped on every push; dataset mounts persist). The mount
# layout has changed before, so scan /kaggle/input instead of hard-coding.
import os
from pathlib import Path

def _read_hf_token():
    base = Path("/kaggle/input")
    hits = sorted(base.rglob("secrets")) if base.is_dir() else []
    for hit in hits:
        for line in hit.read_text().splitlines():
            if line.strip().startswith("HF_TOKEN="):
                return line.split("=", 1)[1].strip().strip('"').strip("'")
    return None

_tok = _read_hf_token()
if _tok:
    os.environ["HF_TOKEN"] = _tok
print("HF auth:", "token found" if _tok else "none (card will NOT be updated)")

In [ ]:
# --- 3. GPU sanity ----------------------------------------------------------
# Fail fast on the wrong accelerator: the API-default P100 (sm_60) has no CUDA
# kernels in Kaggle's torch build; configs pin machine_shape=NvidiaTeslaT4.
assert torch.cuda.is_available(), "No CUDA GPU — check the kernel's accelerator settings"
cap = torch.cuda.get_device_capability(0)
name = torch.cuda.get_device_name(0)
print(f"GPU: {name} (sm_{cap[0]}{cap[1]})")
assert cap >= (7, 0), f"{name} unusable: Kaggle torch ships no kernels for it"
x = torch.randn(256, 256, device="cuda") @ torch.randn(256, 256, device="cuda")
torch.cuda.synchronize()
print("CUDA matmul OK:", float(x.sum()))

In [ ]:
# --- 4. Locate BOTH staged test parts ----------------------------------------
# The full test split is staged as two mutually exclusive halves (Kaggle's
# 20 GB kernel-output cap): sportsmot-test-1.tar (even-indexed sequences) and
# sportsmot-test-2.tar (odd-indexed), mounted via kernel_sources. Both are
# REQUIRED — benchmarking a silent subset would misreport coverage. Each part
# extracts into ITS OWN directory (a shared dir + background extraction once
# deleted part 1 mid-benchmark) and is removed after processing.
import shutil
import subprocess
import time

def _find_tar(part):
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    for pattern in (f"*/sportsmot-test-{part}.tar", f"*/*/sportsmot-test-{part}.tar",
                    f"*/*/*/sportsmot-test-{part}.tar"):
        hits = sorted(base.glob(pattern))
        if hits:
            return hits[0]
    return None

TARS = {part: _find_tar(part) for part in (1, 2)}
for part, tar in TARS.items():
    assert tar is not None, (f"sportsmot-test-{part}.tar not mounted — run "
                             f"testing/prepare-data-{part} and keep both staging "
                             "slugs in this config's kernel_sources")
    print(f"part {part}: {tar}")

def part_dir(part):
    return Path(f"/tmp/sportsmot-part{part}")

def extract_part(part):
    """Extract one staged half into its own directory; returns its seq dirs."""
    dest = part_dir(part)
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    _t = time.time()
    subprocess.run(["tar", "-xf", str(TARS[part]), "-C", str(dest)], check=True)
    seqs = sorted(p for p in (dest / "test").iterdir() if (p / "img1").is_dir())
    print(f"part {part}: extracted {len(seqs)} sequences in {time.time() - _t:.0f}s",
          flush=True)
    return seqs

In [ ]:
# --- 5. Load the fine-tuned model from the Hub ------------------------------
from transformers import AutoImageProcessor, AutoModelForObjectDetection

MODEL_ID = "smallTech/rtdetr-sportsmot"
_t = time.time()
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForObjectDetection.from_pretrained(MODEL_ID).to("cuda").eval()
LOAD_S = time.time() - _t
print(f"loaded {MODEL_ID} in {LOAD_S:.1f}s "
      f"({sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params)")

In [ ]:
# --- 6. Performance benchmark over BOTH parts --------------------------------
# MAX T4 UTILIZATION: a single 640x640 image nowhere near saturates the GPU's
# SMs, so the pipeline batches 8 images per forward pass under fp16
# autocast (Turing tensor cores) — that is how "all the cores" get used.
# Image decode/preprocess overlaps GPU compute via a thread pool (Kaggle GPU
# kernels have 4 vCPUs). Two distinct numbers are measured:
#   * THROUGHPUT — frames/s of the batched, fp16, prefetched pipeline: the
#     utilization-maximizing number;
#   * single-image LATENCY percentiles — a separate batch-1 probe over the
#     first 300 frames, since latency and throughput answer different
#     questions and batching trades the former for the latter.
# The test split has NO ground truth, so accuracy metrics are impossible —
# detections/frame and confidence stats are tracked instead.
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

EVAL_STRIDE = 1
CONF = 0.5
BATCH = 8
PROBE_N = 300

def load_rgb(path):
    return Image.open(path).convert("RGB")

@torch.no_grad()
def run_batch(images):
    """One batched forward: preprocess -> fp16 forward -> postprocess."""
    sizes = torch.tensor([im.size[::-1] for im in images]).to("cuda")
    inputs = processor(images=images, return_tensors="pt").to("cuda")
    with torch.autocast("cuda", dtype=torch.float16):
        outputs = model(**inputs)
    return processor.post_process_object_detection(
        outputs, target_sizes=sizes, threshold=CONF)

# single-image latency probe (batch 1, same fp16 path), then the batched run
from codecarbon import EmissionsTracker
# GPU-compute emissions tracking (codecarbon): measured, not estimated.
_co2 = EmissionsTracker(project_name="rtdetr-sportsmot-testing", log_level="error",
                        save_to_file=False, measure_power_secs=30)
_co2.start()

latencies_1 = []
probe_done = False
det_counts, score_sums = [], []
per_seq = {}
n_frames = 0
gpu_s = 0.0
t_bench = time.time()
# Overlap I/O with compute: extract part 2 in the background while the GPU
# processes part 1 (both halves fit on disk simultaneously; each is deleted
# right after processing).
import threading
_bg = {}
_t2 = threading.Thread(target=lambda: _bg.update({2: extract_part(2)}))
_parts_seqs = {1: extract_part(1)}
_t2.start()
for _part in (1, 2):
    if _part not in _parts_seqs:
        _t2.join()
        _parts_seqs.update(_bg)
    seq_dirs = _parts_seqs[_part]
    for seq_dir in seq_dirs:
        frames = sorted((seq_dir / "img1").glob("*.jpg"))[::EVAL_STRIDE]
        if not probe_done:                 # probe on the first sequence(s)
            for img_path in frames[:PROBE_N - len(latencies_1)]:
                image = load_rgb(img_path)
                torch.cuda.synchronize(); _t0 = time.perf_counter()
                run_batch([image])
                torch.cuda.synchronize()
                latencies_1.append(time.perf_counter() - _t0)
            probe_done = len(latencies_1) >= PROBE_N
        seq_det = []
        with ThreadPoolExecutor(max_workers=4) as pool:
            loaded = pool.map(load_rgb, frames)   # prefetch overlaps GPU work
            batch = []
            def flush(batch):
                torch.cuda.synchronize(); _t0 = time.perf_counter()
                results = run_batch(batch)
                torch.cuda.synchronize()
                dt = time.perf_counter() - _t0
                counts = [len(r["scores"]) for r in results]
                scores = [float(r["scores"].mean()) for r in results if len(r["scores"])]
                return dt, counts, scores
            for image in loaded:
                batch.append(image)
                if len(batch) == BATCH:
                    dt, counts, scores = flush(batch)
                    gpu_s += dt; det_counts += counts; seq_det += counts
                    score_sums += scores; n_frames += len(batch)
                    batch = []
            if batch:
                dt, counts, scores = flush(batch)
                gpu_s += dt; det_counts += counts; seq_det += counts
                score_sums += scores; n_frames += len(batch)
        per_seq[seq_dir.name] = {
            "part": _part, "frames": len(frames),
            "detections_per_frame_mean": sum(seq_det) / max(1, len(seq_det)),
        }
        print(f"{seq_dir.name} (part {_part}): {len(frames)} frames, "
              f"{per_seq[seq_dir.name]['detections_per_frame_mean']:.1f} det/frame, "
              f"cum. throughput {n_frames / gpu_s:.1f} fps", flush=True)
    shutil.rmtree(part_dir(_part))         # done with this half — free the disk
WALL_S = time.time() - t_bench
print(f"\nbenchmarked {n_frames} frames across both parts in {WALL_S / 60:.1f} min "
      f"(GPU-timed throughput {n_frames / gpu_s:.1f} fps)")

In [ ]:
# --- 7. Benchmarks: log + persist --------------------------------------------
import json
import statistics

lat_ms = sorted(l * 1000 for l in latencies_1)
def pct(p):
    return lat_ms[min(len(lat_ms) - 1, int(p / 100 * len(lat_ms)))]

CO2_G = 1000 * (_co2.stop() or 0)              # kg -> grams
benchmarks = {
    "model": MODEL_ID,
    "co2_eq_emissions_g": round(CO2_G, 1),
    "dataset": "Lekim89/sportsmot",
    "split": "test, FULL split via two staged halves "
             "(no public ground truth — performance metrics only)",
    "sequences": len(per_seq),
    "frames_benchmarked": n_frames,
    "eval_stride": EVAL_STRIDE,
    "confidence_threshold": CONF,
    "throughput": {"pipeline": f"batched x{BATCH}, fp16 autocast, 4-thread prefetch",
                    "fps_gpu_timed": n_frames / gpu_s,
                    "fps_wall_clock": n_frames / WALL_S},
    "latency_ms_single_image": {"probe_frames": len(lat_ms),
                                 "mean": statistics.mean(lat_ms), "p50": pct(50),
                                 "p90": pct(90), "p99": pct(99)},
    "detections_per_frame": {"mean": statistics.mean(det_counts),
                              "p50": sorted(det_counts)[len(det_counts) // 2],
                              "max": max(det_counts)},
    "mean_confidence_of_detections": statistics.mean(score_sums),
    # Full environment + method disclosure: performance numbers are only
    # comparable when the hardware, software, and parallelism assumptions
    # they were measured under are stated.
    "environment": {
        "gpu": torch.cuda.get_device_name(0),
        "gpu_compute_capability": f"sm_{cap[0]}{cap[1]}",
        "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
        "cuda": torch.version.cuda,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "python": __import__("platform").python_version(),
        "cpu_cores": __import__("os").cpu_count(),
        "model_load_s": round(LOAD_S, 1),
    },
    "method": {
        "batch_size": BATCH,
        "precision": "fp16 autocast (fp32 weights)",
        "image_prefetch_threads": 4,
        "frame_stride": EVAL_STRIDE,
        "confidence_threshold": CONF,
        "io_overlap": "part-2 tar extracted in a background thread during part-1 compute",
        "timing_scope": "CUDA-synchronized: preprocess + forward + postprocess per batch",
        "latency_probe": f"separate batch-1 pass over the first {PROBE_N} frames",
    },
    "per_sequence": per_seq,
}
out = Path("/kaggle/working/benchmarks.json")
out.write_text(json.dumps(benchmarks, indent=2))
print(json.dumps({k: v for k, v in benchmarks.items() if k != "per_sequence"}, indent=2))
print("=" * 62)
print(f"PERFORMANCE BENCHMARK COMPLETE — throughput "
      f"{benchmarks['throughput']['fps_gpu_timed']:.1f} fps (batched fp16), "
      f"single-image latency p50 {pct(50):.0f} ms, "
      f"{statistics.mean(det_counts):.1f} det/frame @conf>={CONF} "
      f"on {torch.cuda.get_device_name(0)}")
print("=" * 62)

In [ ]:
# --- 8. Model-card section publisher -----------------------------------------
# Replaces (or appends) a marker-delimited section in the Hub model card, so
# re-runs update in place instead of stacking duplicates. A legacy section
# with the same header but no markers is replaced up to the next "## ".
import re
from huggingface_hub import HfApi, hf_hub_download

def publish_card_section(marker: str, header: str, body_md: str, commit: str):
    if not os.environ.get("HF_TOKEN"):
        print("no HF_TOKEN — model card NOT updated "
              "(benchmarks.json is in the kernel output)")
        return
    start, end = f"<!-- {marker}:start -->", f"<!-- {marker}:end -->"
    section = f"{start}\n{body_md.strip()}\n{end}"
    card = Path(hf_hub_download(MODEL_ID, "README.md", force_download=True)).read_text()
    if start in card and end in card:
        card = card[:card.index(start)] + section + card[card.index(end) + len(end):]
    elif header in card:                      # legacy: headed section, no markers
        i = card.index(header)
        m = re.search(r"\n## ", card[i + len(header):])
        j = i + len(header) + m.start() + 1 if m else len(card)
        card = card[:i] + section + "\n" + card[j:]
    else:
        card = card.rstrip() + "\n\n" + section + "\n"
    HfApi().upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md",
                        repo_id=MODEL_ID, commit_message=commit)
    print(f"model card updated: {header!r}")

In [ ]:
# --- 9. Publish to the model card --------------------------------------------
th = benchmarks["throughput"]
lat = benchmarks["latency_ms_single_image"]
det = benchmarks["detections_per_frame"]
env, m = benchmarks["environment"], benchmarks["method"]
body = f"""## Benchmark — test split (performance only)

Measured on the **full SportsMOT `test` split** ({benchmarks['sequences']}
sequences, {benchmarks['frames_benchmarked']:,} frames), staged as two mutually exclusive
halves to fit Kaggle's kernel-output cap. The test split has **no public
ground truth** (withheld for the SportsMOT challenge server), so no accuracy
metrics are possible here — see the val-split benchmark above for those.

| metric ({benchmarks['environment']['gpu']}) | value |
|---|---|
| throughput ({th['pipeline']}) | **{th['fps_gpu_timed']:.1f} fps** (GPU-timed) · {th['fps_wall_clock']:.1f} fps wall-clock |
| single-image latency (batch 1, fp16, {lat['probe_frames']}-frame probe) | mean {lat['mean']:.1f} ms · p50 {lat['p50']:.1f} · p90 {lat['p90']:.1f} · p99 {lat['p99']:.1f} |
| detections per frame (conf ≥ {benchmarks['confidence_threshold']}) | mean {det['mean']:.1f} · p50 {det['p50']} · max {det['max']} |
| mean confidence of detections | {benchmarks['mean_confidence_of_detections']:.3f} |
| model load time | {benchmarks['environment']['model_load_s']} s |
| emissions, this run (codecarbon, measured) | {benchmarks['co2_eq_emissions_g']:.1f} g CO₂eq |

**Environment & method** — the assumptions these numbers were measured under:

| | |
|---|---|
| GPU | {env['gpu']} ({env['gpu_compute_capability']}, {env['gpu_memory_gb']} GB) |
| Software | torch {env['torch']} · CUDA {env['cuda']} · transformers {env['transformers']} · Python {env['python']} |
| CPU | {env['cpu_cores']} vCPUs (image decode / prefetch) |
| Batching & precision | {m['batch_size']} images per forward, {m['precision']} |
| Parallelism | {m['image_prefetch_threads']}-thread image prefetch; {m['io_overlap']} |
| Timing scope | {m['timing_scope']} |
| Latency probe | {m['latency_probe']} |
| Inputs | frame stride {m['frame_stride']}, confidence ≥ {benchmarks['confidence_threshold']} |
"""
publish_card_section("benchmark:test", "## Benchmark — test split", body,
                     "Update test-split performance benchmark (testing kernel)")